# HEAL benchmark evals
- evaluate using openAI API to get baseline evals

In [1]:
# to get new pysqlite3, need to reinstall
# pip install pysqlite3-binary --force-reinstall
import pysqlite3
import sys
sys.modules["sqlite3"] = pysqlite3

In [2]:
import numpy as np
import ast
import pandas as pd
import os
import openai
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
# set which GPU to use
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

### HEAL CDE data

In [4]:
# obtained from here: https://uchicago.app.box.com/file/2056763737745?s=drlzczyp5lhwobznbgcehkzbphm5x20n
target_file = "/opt/gpudata/aartiv/heal_cde/master_cde/master_sde_v2.jsonl"
target_df = pd.read_json(target_file, lines=True)
target_df.head(n=3)

,sde_id,sde_name,sde_description,sde_data_type,sde_permissible_values,sde_pv_description,sde_additional_info,sde_parent_name,sde_parent_description,sde_parent_domain,sde_parent_repository
0,aQO0VmltMn,Address City Name City,The city or township for the address to descri...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH
1,ZqxxvEdcmt,Address County Name County,A region created by territorial division by a ...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH
2,elEZcZ9NdL,Address Line 1,The address where a mail piece is intended to ...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH


In [5]:
target_df.shape

(56558, 11)

In [6]:
target_df['sde_parent_repository'].unique()

array(['NIH', 'HEAL', 'PhenX'], dtype=object)

In [7]:
heal_target_df = target_df[target_df['sde_parent_repository'] == 'HEAL']

In [8]:
heal_target_df.shape

(5072, 11)

In [9]:
heal_target_df.head(n=3)

,sde_id,sde_name,sde_description,sde_data_type,sde_permissible_values,sde_pv_description,sde_additional_info,sde_parent_name,sde_parent_description,sde_parent_domain,sde_parent_repository
22743,Crav3desire,Craving Scale desire to use,Scale describing how strong desire to use was ...,Numeric,0;1;2;3;4;5;6;7;8;9,0 = No desire;;;;;;;;;9=Strong desire,Additional Notes (Question Text): Please rate ...,craving-scale-3-item-cde.xlsx,The Craving Scale is a 3-item measure of cravi...,Substance Use,HEAL
22744,Crav3likely,Craving Scale likelihood of use,Scale describing how likely participant is to ...,Numeric,0;1;2;3;4;5;6;7;8;9,0 = No likelihood;;;;;;;;;9=Strong likelihood,Additional Notes (Question Text): Please imagi...,craving-scale-3-item-cde.xlsx,The Craving Scale is a 3-item measure of cravi...,Substance Use,HEAL
22745,Crav3urge,Craving Scale urges,Scale describing how strong is the urge for dr...,Numeric,0;1;2;3;4;5;6;7;8;9,0 = No urge;;;;;;;;;9=Strong urge,Additional Notes (Question Text): Please rate ...,craving-scale-3-item-cde.xlsx,The Craving Scale is a 3-item measure of cravi...,Substance Use,HEAL


In [10]:
heal_target_df.iloc[0]['sde_pv_description']

'0 = No desire;;;;;;;;;9=Strong desire'

### HEAL benchmarks
- created by Brienna
- /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv
- /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv
- see cleaned version here https://uchicago.app.box.com/file/2076961773404

### format name, desc and values 
- similar to training

In [11]:
def preprocess_variable(row):
    name = row.get(f'sde_name') or "[NO_NAME]"
    desc = row.get(f'sde_description') or "[NO_DESC]"
    raw_values = row.get(f'sde_permissible_values')

    if raw_values:
        values_list = [v.strip() for v in raw_values.split(";") if v.strip()]
        value_descriptions = row.get(f'sde_pv_description')
        if value_descriptions:
            values_descriptions_list = [v.strip() for v in value_descriptions.split(";") if v.strip()]
        else:
            values_descriptions_list = []
        value = ", ".join(values_list + values_descriptions_list) if values_list else "[NO_VALUE]"
    else:
        value = "[NO_VALUE]"

    return f"{name} {desc} {value}"

In [12]:
heal_target_df["name_desc_val"] = heal_target_df.apply(preprocess_variable, axis=1)

/tmp/ipykernel_1472712/1442155599.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  heal_target_df["name_desc_val"] = heal_target_df.apply(preprocess_variable, axis=1)


In [13]:
heal_target_df.shape

(5072, 12)

### Create embeddings

In [24]:
load_dotenv() 
openai.api_key = os.getenv("OPENAI_API_KEY")
if openai.api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable not set!")

In [ ]:
# see https://developers.openai.com/api/docs/guides/embeddings#use-cases
from openai import OpenAI
client = OpenAI()

def get_embedding(text, modelname):
    text = text.replace("\n", " ")
    return client.embeddings.create(input = [text], model=modelname).data[0].embedding

### Embedding the heal target df takes ~17 mins to complete for heal df with 5k rows
- may be time can be further reduced using openAI batch API?
- cost = $0.01 for 5,072 requests
- embedding the name_desc_val field

In [26]:
heal_target_df['text_embedding_3_small'] = heal_target_df.name_desc_val.apply(lambda x: get_embedding(x, modelname='text-embedding-3-small'))
heal_target_df.to_csv('heal_df.embedded.text_embedding_3_small.csv', index=False)

In [14]:
heal_target_embedded = pd.read_csv('heal_df.embedded.text_embedding_3_small.csv')
heal_target_embedded['target_array_embedding'] = heal_target_embedded.text_embedding_3_small.apply(eval).apply(np.array)

In [15]:
heal_target_embedded.head(n=3)

,sde_id,sde_name,sde_description,sde_data_type,sde_permissible_values,sde_pv_description,sde_additional_info,sde_parent_name,sde_parent_description,sde_parent_domain,sde_parent_repository,name_desc_val,text_embedding_3_small,target_array_embedding
0,Crav3desire,Craving Scale desire to use,Scale describing how strong desire to use was ...,Numeric,0;1;2;3;4;5;6;7;8;9,0 = No desire;;;;;;;;;9=Strong desire,Additional Notes (Question Text): Please rate ...,craving-scale-3-item-cde.xlsx,The Craving Scale is a 3-item measure of cravi...,Substance Use,HEAL,Craving Scale desire to use Scale describing h...,"[-0.017852783203125, 0.036224365234375, -0.022...","[-0.017852783203125, 0.036224365234375, -0.022..."
1,Crav3likely,Craving Scale likelihood of use,Scale describing how likely participant is to ...,Numeric,0;1;2;3;4;5;6;7;8;9,0 = No likelihood;;;;;;;;;9=Strong likelihood,Additional Notes (Question Text): Please imagi...,craving-scale-3-item-cde.xlsx,The Craving Scale is a 3-item measure of cravi...,Substance Use,HEAL,Craving Scale likelihood of use Scale describi...,"[-0.037490781396627426, 0.028369257226586342, ...","[-0.037490781396627426, 0.028369257226586342, ..."
2,Crav3urge,Craving Scale urges,Scale describing how strong is the urge for dr...,Numeric,0;1;2;3;4;5;6;7;8;9,0 = No urge;;;;;;;;;9=Strong urge,Additional Notes (Question Text): Please rate ...,craving-scale-3-item-cde.xlsx,The Craving Scale is a 3-item measure of cravi...,Substance Use,HEAL,Craving Scale urges Scale describing how stron...,"[-0.0238800048828125, 0.023162841796875, -0.01...","[-0.0238800048828125, 0.023162841796875, -0.01..."


In [60]:
heal_target_embedded.to_csv('/opt/gpudata/aartiv/heal_cde/master_cde/heal_df.array_embedding.text_embedding_3_small.csv', index=False)

In [16]:
heal_target_embedded.shape

(5072, 14)

### Process HEAL benchmarks

In [17]:
def preprocess_variable_benchmark(row):
    name = row.get(f'field_title') or "[NO_NAME]"
    desc = row.get(f'field_description') or "[NO_DESC]"
    raw_values = row.get(f'field_enumLabels')

    if isinstance(raw_values, dict):
        value_labels = ','.join([f'{k}={v}' for k, v in raw_values.items()])
        value_constraints = ','.join(row.get(f'field_constraints'))
        value = ';'.join([value_constraints, value_labels])
        
    else:
        value = "[NO_VALUE]"

    return f"{name} {desc} {value}"

In [17]:
benchmark_file_list = ['/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv',
                       '/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv']

In [35]:
for benchmark_name in benchmark_file_list:
    print(f'processing {benchmark_name}')
    output_file = benchmark_name.split('.')[0] + '.processed.csv'
    df = pd.read_csv(benchmark_name)
    df['field_enumLabels'] = df['field_enumLabels'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df['field_constraints'] = df['field_constraints'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df['name_desc_val'] = df.apply(preprocess_variable_benchmark, axis=1)
    df.to_csv(output_file)
    

processing /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv
processing /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv


In [18]:
processed_benchmark_file_list = ['/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.processed.csv',
                       '/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.processed.csv']

### Embed processed HEAL benchmarks
- using openai embedding model
- takes ~30 secs to embed both benchmarks
- total spend: target df + benchmarks = $0.01

In [43]:
for benchmark_name in processed_benchmark_file_list:
    print(f'embedding {benchmark_name}')
    output_file = benchmark_name.split('.')[0] + '.embedded.text_embedding_3_small.csv'
    df = pd.read_csv(benchmark_name)
    df['text_embedding_3_small'] = df.name_desc_val.apply(lambda x: get_embedding(x, modelname='text-embedding-3-small'))
    df.to_csv(output_file, index=False)

embedding /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.processed.csv
embedding /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.processed.csv


### Embedded benchmarks file list

In [23]:
embedded_benchmark_file_list = ['/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.embedded.text_embedding_3_small.csv',
                       '/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.embedded.text_embedding_3_small.csv']

# get top k

In [19]:
heal_cde_matrix = np.vstack(heal_target_embedded["target_array_embedding"].values)

In [20]:
def get_top_k(sim_matrix, k):
    top_res = np.argsort(-sim_matrix, axis=1)[:, :k]
    # print(f'{k}: {top_res}')
    top_res_series = pd.Series(list(top_res))
    return top_res_series

In [21]:
def index_to_name(index_list):
    # print(index_list)
    name_list = []
    for index in index_list:
        name = heal_target_embedded.loc[int(index)]["sde_name"]
        # print(f'name: {name}')
        name_list.append(name)
    return name_list


# normalize spaces etc for truth comparison
def normalize(s):
    return " ".join(s.split())


def get_top_k_benchmark(embedded_benchmark_name):
    # print(f'processing {embedded_benchmark_name}')
    df = pd.read_csv(embedded_benchmark_name, index_col=0)
    # print(df['text_embedding_3_small'])
    df['array_embedding'] = df['text_embedding_3_small'].apply(ast.literal_eval).apply(np.array)
    # df['array_embedding'] = df.text_embedding_3_small.apply(eval).apply(np.array)
    query_matrix = np.vstack(df["array_embedding"].values)
    sim_matrix = cosine_similarity(query_matrix, heal_cde_matrix)

    # print(query_matrix.shape)
    top_k_list = [1, 5, 10]
    for k in top_k_list:
        col_name = f'top_{k}_results'
        df[col_name] = get_top_k(sim_matrix, k=k)
        # print(f'{df[col_name]}')
        # print(f'{col_name}: {df[col_name]}')
        target_col_name = f'top_{k}_names'
        df[target_col_name] = df[col_name].apply(index_to_name)
        # print(f'{target_col_name}: {df[target_col_name]}')

    print('returning top k results')
    return df



def calculate_acc(truth, pred):
  correct = 0
  for t, p_list_of_names in zip(truth, pred):
      if t in p_list_of_names:
          correct += 1
  return correct / len(truth)




def run_evals(df, benchmark_name, embedding_model):
    # print('calculating metrics')
    evals = {}
    row_index = []
    top_k_list = [1, 5, 10]
    for k in top_k_list:
        evals[f'accuracy_{k}'] = []

    row_index.append(f'{benchmark_name}_{df.shape[0]}_{embedding_model}')
    for k in top_k_list:
        col_name = f'top_{k}_names'
        top_k_names_list = df[col_name].to_list()
        truth_list = df['element_title'].to_list()
        accuracy = calculate_acc(truth_list, top_k_names_list)
        evals[f'accuracy_{k}'].append(accuracy)

    # print('returning metrics')
    return pd.DataFrame(evals, index=row_index)



def run_evals_per_row(row, k):
    col_name = f'top_{k}_names'
    top_k_names_list = row[col_name]
    truth = row['element_title']
    # print(f'truth: {truth}')
    # print(f'top_k_names: {top_k_names_list}')
    try:
        is_match = normalize(truth) in [normalize(x) for x in top_k_names_list]
    except Exception as e:
        print(f'in exception! truth: {truth}, top_k_names_list: {top_k_names_list}')
        print(f'original row: {row}')
        is_match = truth in top_k_names_list
    return is_match


def get_metrics_per_row(df):
    top_k_list = [1, 5, 10]
    for k in top_k_list:
        output_cols = f'is_match_in_top_{k}_name_desc'
        df[output_cols] = df.apply(lambda x: run_evals_per_row(x, k), axis=1)
    return df

In [24]:
embedding_model_list = ['text_embedding_3_small']
metrics_per_row_results = pd.concat([
    get_metrics_per_row(df=get_top_k_benchmark(f))
    for f in embedded_benchmark_file_list
])

returning top k results
returning top k results
in exception! truth: nan, top_k_names_list: ['Race and/or Ethnicity']
original row: Unnamed: 0                                                                  388
field_name                                                             ethnic_p
field_type                                                              integer
field_title                                             What is your ethnicity?
field_enumLabels              {'1': 'Hispanic or Latino', '2': 'Not Hispanic...
field_constraints                                          ['1', '2', '3', '4']
field_description                                     : What is your ethnicity?
CDE_instrument                                                     Demographics
element_name                                                            HI_LA_p
element_type                                                                NaN
element_title                                                       

In [25]:
metrics_per_row_results.shape

(138, 32)

In [26]:
metrics_per_row_results.head()

,field_name,field_type,field_title,field_enumLabels,field_constraints,field_description,CDE_instrument,element_name,element_type,element_title,...,top_1_names,top_5_results,top_5_names,top_10_results,top_10_names,is_match_in_top_1_name_desc,is_match_in_top_5_name_desc,is_match_in_top_10_name_desc,Unnamed: 0,standardsMappings.id or ID
0,dob,date,Date of birth,NaN,NaN,Contact Information: Date of birth,Demographics,BRTHDTC,string,Birth date,...,[Birth date],"[4970, 5005, 1843, 4971, 4972]","[Birth date, Birth date (child), The World Hea...","[4970, 5005, 1843, 4971, 4972, 5006, 5012, 152...","[Birth date, Birth date (child), The World Hea...",True,True,True,NaN,NaN
1,race___0,boolean,Race: American Indian/Alaska Native,"{'0': 'Unchecked', '1': 'Checked'}","['0', '1']",Contact Information: Race[choice=American Indi...,Demographics,AI_AN\n\n\n\n\n\n,integer,Race and/or Ethnicity,...,[Race and/or Ethnicity],"[4976, 5011, 5015, 4981, 4983]","[Race and/or Ethnicity, Race and/or Ethnicity ...","[4976, 5011, 5015, 4981, 4983, 4977, 4984, 498...","[Race and/or Ethnicity, Race and/or Ethnicity ...",True,True,True,NaN,NaN
2,race___1,boolean,Race: Asian,"{'0': 'Unchecked', '1': 'Checked'}","['0', '1']",Contact Information: Race[choice=Asian],Demographics,Asian,integer,Race and/or Ethnicity,...,[Race and/or Ethnicity],"[4977, 4983, 4984, 4982, 4981]","[Race and/or Ethnicity, Race and/or Ethnicity,...","[4977, 4983, 4984, 4982, 4981, 4978, 4976, 501...","[Race and/or Ethnicity, Race and/or Ethnicity,...",True,True,True,NaN,NaN
3,race___2,boolean,Race: Black or African American,"{'0': 'Unchecked', '1': 'Checked'}","['0', '1']",Contact Information: Race[choice=Black or Afri...,Demographics,Bl_AA,integer,Race and/or Ethnicity,...,[Race and/or Ethnicity],"[4978, 4983, 4982, 4984, 4976]","[Race and/or Ethnicity, Race and/or Ethnicity,...","[4978, 4983, 4982, 4984, 4976, 5011, 4979, 498...","[Race and/or Ethnicity, Race and/or Ethnicity,...",True,True,True,NaN,NaN
4,race___3,boolean,Race: Native Hawaiian or Other Pacific Islander,"{'0': 'Unchecked', '1': 'Checked'}","['0', '1']",Contact Information: Race[choice=Native Hawaii...,Demographics,NH_PI,integer,Race and/or Ethnicity,...,[Race and/or Ethnicity],"[4981, 4976, 5011, 4977, 5015]","[Race and/or Ethnicity, Race and/or Ethnicity,...","[4981, 4976, 5011, 4977, 5015, 4983, 4979, 498...","[Race and/or Ethnicity, Race and/or Ethnicity,...",True,True,True,NaN,NaN


In [27]:
metrics_per_row_results.columns

Index(['field_name', 'field_type', 'field_title', 'field_enumLabels',
       'field_constraints', 'field_description', 'CDE_instrument',
       'element_name', 'element_type', 'element_title', 'element_description',
       'enumLabels', 'encoding', 'constraints.enum', 'standardsMappings.id',
       'standardsMappings.source', 'CDE_HEAL_ID', 'Notes', 'name_desc_val',
       'text_embedding_3_small', 'array_embedding', 'top_1_results',
       'top_1_names', 'top_5_results', 'top_5_names', 'top_10_results',
       'top_10_names', 'is_match_in_top_1_name_desc',
       'is_match_in_top_5_name_desc', 'is_match_in_top_10_name_desc',
       'Unnamed: 0', 'standardsMappings.id or ID'],
      dtype='object')

In [28]:
metrics_per_row_results.to_csv('/opt/gpudata/aartiv/heal_cde/openai.metrics_per_row.csv')

In [29]:
metrics_per_row_results[~metrics_per_row_results['is_match_in_top_1_name_desc']].shape

(68, 32)

In [30]:
embedding_model_list = ['text_embedding_3_small']
results = pd.concat([
    run_evals(df=get_top_k_benchmark(f), benchmark_name=f, embedding_model=emb_model)
    for f in embedded_benchmark_file_list
    for emb_model in embedding_model_list
])

returning top k results
returning top k results


In [31]:
results

,accuracy_1,accuracy_5,accuracy_10
/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.embedded.text_embedding_3_small.csv_48_text_embedding_3_small,0.437500,0.729167,0.875000
/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.embedded.text_embedding_3_small.csv_90_text_embedding_3_small,0.522222,0.900000,0.944444


In [34]:
results.to_csv('openai.results.csv')

In [32]:
results.to_csv('/opt/gpudata/aartiv/heal_cde/openai.results.csv')